two sanity checks:
- stable-only subset: prevents “label flip” cases from corrupting faithfulness/robustness interpretation.
- random attribution baseline (shuffle): ensures the metric doesn’t give “good” scores to nonsense maps.

## Setup

In [1]:

from pathlib import Path
import sys
import os

os.environ["PYTHONHASHSEED"] = "51"

In [2]:

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    project_path = Path("/content/drive/MyDrive/Seminararbeit_KI4Bio") 
    %cd {project_path}

    os.environ["DATA"] = str(project_path / "data")
    os.environ["RESULTS_ROOT"]      = str(project_path / "results")
    os.environ["RUNS_ROOT"]         = str(project_path / "experiments")                   

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
# Install dependencies
##%%capture
%pip install --no-cache-dir -r requirements.txt
%pip install --no-cache-dir -r requirements-captum-nodeps.txt --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 6.3 MB/s  0:00:036.4 MB/s eta 0:00:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [1223 lines of output]
      + /opt/anaconda3/envs/aCV/bin/python /private/var/folders/37/np1g3dns41b8h9lt70j7673m0000gn/T/pip-install-q4poxqyw/numpy_8e7de0e82aad47e8be36dc91e803369d/vendored-meson/meson/meson.py setup /private/var/folders/37/np1g3dns41b8h9lt70j7673m0000gn/T/pip-install-q4poxqyw/numpy_8e7de0e82aad47e8be36dc91e803369d /private/var/folders/37/np1g3dns41b8h9lt70j7673m0000gn/T/pip-install-q4poxqyw/numpy_8e7de0e82aad47e8be36dc91e803369d/.mesonpy-92gazzqu -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=/private/var/folders/37/np1g3dns41b8h9lt70j7673m000

In [4]:
!pip install quantus

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np

import quantus

/opt/anaconda3/envs/aCV/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from src.config import PATHS, DEVICE, SEED, N_PAIRS, IG_STEPS
from src.utils import set_seeds
from src.models import load_model

In [5]:
set_seeds(SEED)

Seeds set to 51 (deterministic=True)


In [ ]:
resnet_model = load_model()

In [ ]:
def expand_saliency_to_channels(a_hw: np.ndarray, x: np.ndarray) -> np.ndarray:
    # a_hw: [N,H,W], x: [N,C,H,W]
    return np.repeat(a_hw[:, None, :, :], repeats=x.shape[1], axis=1)

In [15]:
def denorm_to_01(x_chw: torch.Tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD) -> torch.Tensor:
    """
    x_chw: torch [N,3,H,W] in normalized space
    returns: torch [N,3,H,W] in [0,1]
    """
    m = torch.tensor(mean, device=x_chw.device).view(1,3,1,1)
    s = torch.tensor(std,  device=x_chw.device).view(1,3,1,1)
    x = x_chw * s + m
    return x.clamp(0.0, 1.0)

# Stable-only + random baseline helpers

In [ ]:
def apply_mask_np(x, mask):
    # mask: torch bool [N] or np bool [N]
    if torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy().astype(bool)
    return x[mask]

def shuffle_attributions_per_sample(a):
    """
    a: np array [N,C,H,W] (or [N,H,W]).
    Shuffles values within each sample (destroys spatial structure but keeps histogram).
    """
    a_shuf = a.copy()
    N = a_shuf.shape[0]
    for i in range(N):
        flat = a_shuf[i].reshape(-1)
        np.random.shuffle(flat)
        a_shuf[i] = flat.reshape(a_shuf[i].shape)
    return a_shuf


# Quantus computation

In [ ]:
def compute_quantus_two_metrics(
    model,
    x_t: torch.Tensor,            # [N,3,32,32] normalized
    pred_target: torch.Tensor,    # [N] (use pred_clean)
    sal_hw: torch.Tensor,         # [N,32,32] (clean or corr)
    mean, std,
    device,
    stable_mask: torch.Tensor | None = None,
    seed: int = 0,
):
    rng = np.random.default_rng(seed)

    # denorm to [0,1] for "black" baseline semantics
    x01 = denorm_to_01(x_t, mean, std).detach().cpu().numpy().astype(np.float32)  # [N,3,32,32]
    y   = pred_target.detach().cpu().numpy().astype(int)                          # [N]
    a   = sal_hw.detach().cpu().numpy().astype(np.float32)                        # [N,32,32]
    a   = expand_saliency_to_channels(a, x01)                                     # [N,3,32,32]

    # Stable-only slice (optional)
    if stable_mask is not None:
        x01 = apply_mask_np(x01, stable_mask)
        y   = apply_mask_np(y, stable_mask)
        a   = apply_mask_np(a, stable_mask)

    # --- Metrics ---
    faith = quantus.FaithfulnessCorrelation(
        nr_runs=25,          # start smaller than 100
        subset_size=224,
        perturb_baseline="black",
        abs=True,
        normalise=True,
        return_aggregate=False,   # get per-sample list, then aggregate yourself
        disable_warnings=True,
    )

    sens = quantus.AvgSensitivity(
        nr_samples=50,       # start smaller than 200
        abs=True,
        normalise=False,
        return_aggregate=False,
        disable_warnings=True,
        return_nan_when_prediction_changes=True,  # safer under noise
    )

    faith_scores = faith(model=model, x_batch=x01, y_batch=y, a_batch=a, device=str(device))
    sens_scores  = sens(model=model,  x_batch=x01, y_batch=y, a_batch=a, device=str(device))

    # Random attribution baseline (shuffle)
    a_rand = shuffle_attributions_per_sample(a)
    faith_rand = faith(model=model, x_batch=x01, y_batch=y, a_batch=a_rand, device=str(device))
    sens_rand  = sens(model=model,  x_batch=x01, y_batch=y, a_batch=a_rand, device=str(device))

    def agg(scores):
        scores = np.array(scores, dtype=np.float32)
        return float(np.nanmean(scores)), float(np.nanstd(scores))

    out = {
        "faith_corr_mean": agg(faith_scores)[0],
        "faith_corr_sd":   agg(faith_scores)[1],
        "sens_mean":       agg(sens_scores)[0],
        "sens_sd":         agg(sens_scores)[1],
        "faith_corr_mean_rand": agg(faith_rand)[0],
        "faith_corr_sd_rand":   agg(faith_rand)[1],
        "sens_mean_rand":       agg(sens_rand)[0],
        "sens_sd_rand":         agg(sens_rand)[1],
        "n_eval": int(len(y)),
    }
    return out


In [16]:
deletion_auc = quantus.PixelFlipping(
    features_in_step=32,          # deletes 32 pixels per step (tune)
    perturb_baseline="black",
    abs=True,
    normalise=True,
    return_auc_per_sample=True,   # <-- gives scalar AUC per sample
    disable_warnings=True,
)


In [17]:
# --- Q1 on CLEAN ---
X_q = denorm_to_01(X_clean_t).detach().cpu().numpy().astype(np.float32)   # [N,3,32,32] in [0,1]
y_q = pred_clean.detach().cpu().numpy().astype(int)                      # [N]
a_q = sal_clean.detach().cpu().numpy().astype(np.float32)                # [N,32,32]

q1_auc_per_sample = deletion_auc(
    model=resnet_model,
    x_batch=X_q,
    y_batch=y_q,
    a_batch=a_q,
    softmax=False,                 
    device=str(DEVICE),
    channel_first=True,
)

q1_clean_mean = float(np.mean(q1_auc_per_sample))
q1_clean_std  = float(np.std(q1_auc_per_sample))
print("Q1 Deletion AUC (clean):", q1_clean_mean, "±", q1_clean_std)


Q1 Deletion AUC (clean): 43.19914806127548 ± 115.27792200427061


In [ ]:
# --- Q1 on CORRUPTED ---
xq_corr = denorm_to_01(X_corr_t).detach().cpu().numpy().astype(np.float32)
yq_corr = pred_clean.detach().cpu().numpy().astype(int)   # keep target policy consistent with your setup
aq_corr = sal_corr.detach().cpu().numpy().astype(np.float32)

q1_corr_auc = deletion_auc(
    model=resnet_model,
    x_batch=xq_corr,
    y_batch=yq_corr,
    a_batch=aq_corr,
    softmax=False,
    device=str(DEVICE),
    channel_first=True,
)

row["q1_deletion_auc_mean"] = float(np.mean(q1_corr_auc))
row["q1_deletion_auc_std"]  = float(np.std(q1_corr_auc))


In [ ]:
# pick a metric
faith = quantus.FaithfulnessCorrelation(
    nr_runs=5,          # repeat with different random subsets
    subset_size=224,    # number of features/pixels perturbed per run (tune)
    perturb_baseline="black",
    softmax=False,
    abs=True,
    normalise=True,
)

# inside your loop, after you have X_clean (tensor or numpy), y_clean, sal_clean:
# Quantus expects numpy:
x_batch = X_clean.detach().cpu().numpy()          # [N,3,32,32] float (normalized is ok if consistent)
y_batch = pred_clean.detach().cpu().numpy()       # [N]
a_batch = sal_clean.detach().cpu().numpy()        # [N,32,32]  (your wrapper already makes it 2D)

scores = faith(model=model, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, device=DEVICE)
faith_mean = float(np.mean(scores))
faith_std  = float(np.std(scores))
